In [ ]:
import torch
import math
import torch.optim as optim
from torch.utils.data import DataLoader
from scripts.custom_dataset import CustomDataset
from model.model import MHAModel
from scripts.tokenizer import train_bpe_tokenizer
from scripts.vectorizer import Vectorizer
from scripts.vocabulary import Vocabulary
import pandas as pd
import json
import os

In [ ]:
UNK_TOKEN = '<UNK>'
MASK_TOKEN = '<MASK>'
PAD_TOKEN = '<PAD>'
BOS_TOKEN='<BOS>'
EOS_TOKEN = '<EOS>'
ADD_BOS_EOS_TOKENS = False

USE_CLASSES_WEIGHTS = False
CLASSES_WEIGHTS_SCALER = 12

SHUFFLE = True
DROP_LAST = True
EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5

USE_PRETRAINED = True

BATCH_SIZE = 128
INIT_WEIGHTS = True
BIAS = True
TOKENS_EMBEDDING_DIM = 256
LETTERS_EMBEDDING_DIM = 32 # Важно, если WORD_REPRESENTATION = 'both', то ATTENTION_DIM = (LETTERS_EMBEDDING_DIM * MAX_LETTERS_COUNT) + TOKENS_EMBEDDING_DIM
LETTERS_IN_WORD_ATTENTION_DIM = 128
MAIN_ATTENTION_DIM = 512
MAIN_NUM_HEADS = 8
MAIN_NUM_ENCODER_LAYERS = 4
MAIN_ENCODER_FC_HIDDEN_DIM = MAIN_ATTENTION_DIM*4 # Как в классическом трансформере

CLASSIFIER_FC_HIDDEN_DIM = MAIN_ATTENTION_DIM*2

WORD_REPRESENTATION = 'tokens' # tokens; letters; both  Уровень представления слова (токены, буквы, токены + буквы)
WORDS_POS_ENCODING = 'learnable' # Допустимые значения: sin; learnable; None
TOKENS_POS_ENCODING = None # Допустимые значения: sin; learnable; None
WORD_SUBTOKENS_POS_ENCODING = 'learnable' # Допустимые значения: learnable; None
LETTERS_POS_ENCODING = 'learnable' # Допустимые значения: learnable; sin; None

SUBTOKENS_AGGREGATION = 'attention' # Допустимые значения: mean; sum; attention
AGGREGATION_MOMENT = 'early' # Допустимые значения: early; late

DROPOUT = 0.2
TEMPERATURE = 1
BATCH_FIRST = True

MODEL_SAVE_FILEPATH = f'model/{WORD_REPRESENTATION}_model_params.pt'
DATASET_VERSION = 2.16 # Допустимые занчения: 2.3; 2.16 | В версии 2.3 меньше тренировочных примеров, по сравнению с 2.16. Точность на тестовой выборке практически не меняется
DATASET_PATH = f'D:/Files/Datasets/UD_Russian-SynTagRus-master_{DATASET_VERSION}'
CORPUS_TEXTS_FILEPATH = f'data/corpus_{DATASET_VERSION}.txt'

VOCABULARY_SIZE = 10000
MIN_FRECQUENCY_PAIR = 240

MAX_WORDS_COUNT = 0
MAX_TOKENS_COUNT = 0
MAX_WORD_SUBTOKENS_COUNT = 70
MAX_LETTERS_COUNT = 12


RANDOM_STATE = 42

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# DEVICE = 'cpu'

In [3]:
print(torch.backends.cuda.flash_sdp_enabled())
print(torch.backends.cuda.mem_efficient_sdp_enabled())
print(torch.backends.cuda.math_sdp_enabled())

True
True
True


In [4]:
def find_max_words_source_len(dataframe:pd.DataFrame)->int:
    '''Возвращает максимальную длину НЕ ТОКЕНИЗИРОВАННОЙ входной последовательности в датафрейме'''
    max_words_tokens = 0
    for i in range(len(dataframe)):
        max_words_tokens = max(len(dataframe.loc[i, 'source_words']), max_words_tokens)
    return max_words_tokens

In [5]:
def find_max_tokens_source_len(dataframe:pd.DataFrame, tokenizer)->int:
    '''Возвращает максимальную длину ТОКЕНИЗИРОВАННОЙ входной последовательности в датафрейме'''
    max_source_tokens = 0
    for i in range(len(dataframe)):
        max_source_tokens = max(len(tokenizer.encode(dataframe.loc[i, 'source_text']).tokens), max_source_tokens)
    return max_source_tokens

In [6]:
def generate_batches(dataset:CustomDataset, batch_size:int, shuffle:bool=True, drop_last:bool=True, device='cpu'):
    '''Создает батчи из датасета и переносит данные на девайс.'''
    dataloader = DataLoader(dataset, batch_size, shuffle, drop_last=drop_last)
    for data_dict in dataloader:
        out_data_dict = {}
        for name, _ in data_dict.items():
            out_data_dict[name] = data_dict[name].to(device)
        yield out_data_dict

In [ ]:
def save_results_to_file(model, model_filepath:str, train_states:list=None, validation_states:list=None):
    '''Сохраняет параметры модели и метрики обучения в файлы.'''
    torch.save(model, model_filepath)
    if train_states is not None:
        with open(f"data/{WORD_REPRESENTATION}_train_states.json", "w", encoding="utf-8") as file:
            json.dump(train_states, file, indent=4, ensure_ascii=False)
        
        with open(f"model/{WORD_REPRESENTATION}_model_hyperparams.json", "w", encoding="utf-8") as file:
            json.dump(model.get_hyperparams(), file, indent=4, ensure_ascii=False)

    if validation_states is not None:
        with open(f"data/{WORD_REPRESENTATION}_validation_states.json", "w", encoding="utf-8") as file:
            json.dump(validation_states, file, indent=4, ensure_ascii=False)

In [8]:
def normalize_sizes(predictions:dict[str:torch.tensor], targets:dict[str:torch.tensor], target_names:list[str]):
    for key in target_names:
        # Для predictions: [B, S, C] -> [B*S, C]
        if len(predictions[key].size()) == 3:
            predictions[key] = predictions[key].contiguous().view(-1, predictions[key].size(-1))
        
        # Для targets: [B, S] -> [B*S]
        if len(targets[key].size()) == 2:
            targets[key] = targets[key].contiguous().view(-1)
    
    return predictions, targets

In [9]:
def reveal_mistakes(predictions:dict[str:torch.tensor], targets:dict[str:torch.tensor], target_names:list[str], pad_idx:int=0) -> dict[str:torch.Tensor]:
    incorrect_indices = {}
    for key in target_names:
        _, pred_indices = predictions[key].max(dim=-1)
        correct_indices = torch.eq(pred_indices, targets[key])
        valid_indices = torch.ne(targets[key], pad_idx)
        incorrect_indices[key] = (~(correct_indices*valid_indices))*valid_indices  # bool tensor
    return incorrect_indices

In [10]:
def compute_loss(predictions:dict[str:torch.tensor], targets:dict[str:list[int]], target_names:list[str], target_weights:dict[str:float], pad_idx:int=0):
    predictions, targets = normalize_sizes(predictions, targets, target_names)
    losses = {}
    total_loss = 0
    for key in target_names:
        target_weights[key]['classes_weights'] = target_weights[key]['classes_weights'].to(DEVICE)
        losses[key] = torch.nn.functional.cross_entropy(predictions[key], targets[key], weight=target_weights[key]['classes_weights'], ignore_index=pad_idx)
        total_loss += losses[key] * target_weights[key]['grammem_weight']

    return total_loss, losses

In [11]:
def compute_accuracy(predictions:dict[str:torch.tensor], targets:dict[str:list[int]], target_names:list[str], pad_idx:int=0)->dict[str:float]:
    predictions, targets = normalize_sizes(predictions, targets, target_names)
    
    accuracies = {}
    for key in target_names:
        _, pred_indices = predictions[key].max(dim=1)
        
        correct_indices = torch.eq(pred_indices, targets[key]).float()
        valid_indices = torch.ne(targets[key], pad_idx).float()
        
        n_correct = (correct_indices * valid_indices).sum().item()
        n_valid = valid_indices.sum().item()
        accuracies[key] = n_correct / n_valid

    return accuracies

In [12]:
def inverse_weights(data:dict[str:float], alpha:float):
    inversed_data = {}
    for key in data.keys():
        inversed_data[key] = 1 * alpha / data[key]
    
    return inversed_data

In [13]:
train_df = pd.read_parquet(os.path.join(DATASET_PATH, 'ru_syntagrus-ud-train.parquet'))
validation_df = pd.read_parquet(os.path.join(DATASET_PATH, 'ru_syntagrus-ud-dev.parquet'))
test_df = pd.read_parquet(os.path.join(DATASET_PATH, 'ru_syntagrus-ud-test.parquet'))

In [14]:
train_df.head(5)

,source_text,source_words,lemmas,upos,xpos,feats,head,deprel,misc
0,Анкета.,"[Анкета, .]","[анкета, .]","[NOUN, PUNCT]","[None, None]","[{'Animacy': 'Inan', 'Case': 'Nom', 'Gender': ...","[0, 1]","[root, punct]","[{'SpaceAfter': 'No'}, None]"
1,Начальник областного управления связи Семен Ер...,"[Начальник, областного, управления, связи, Сем...","[начальник, областной, управление, связь, Семе...","[NOUN, ADJ, NOUN, NOUN, PROPN, PROPN, AUX, NOU...","[None, None, None, None, None, None, None, Non...","[{'Animacy': 'Anim', 'Case': 'Nom', 'Gender': ...","[8, 3, 1, 3, 1, 5, 8, 0, 8, 11, 8, 13, 11, 11,...","[nsubj, amod, nmod, nmod, appos, flat:name, co...","[None, None, None, None, None, None, None, Non..."
2,"В приемной его с утра ожидали посетители, - ко...","[В, приемной, его, с, утра, ожидали, посетител...","[в, приемная, он, с, утро, ожидать, посетитель...","[ADP, NOUN, PRON, ADP, NOUN, VERB, NOUN, PUNCT...","[None, None, None, None, None, None, None, Non...","[None, {'Animacy': 'Inan', 'Case': 'Loc', 'Gen...","[2, 6, 6, 5, 6, 0, 6, 13, 13, 13, 13, 13, 7, 1...","[case, obl, obj, case, obl, root, nsubj, punct...","[None, None, None, None, None, None, {'SpaceAf..."
3,Однако стиль работы Семена Еремеевича заключал...,"[Однако, стиль, работы, Семена, Еремеевича, за...","[однако, стиль, работа, Семен, Еремеевич, закл...","[ADV, NOUN, NOUN, PROPN, PROPN, VERB, ADP, PRO...","[None, None, None, None, None, None, None, Non...","[{'Degree': 'Pos'}, {'Animacy': 'Inan', 'Case'...","[6, 6, 2, 3, 4, 0, 8, 6, 11, 11, 8, 13, 11, 16...","[advmod, nsubj, nmod, nmod, flat:name, root, c...","[None, None, None, None, None, None, None, {'S..."
4,"Приемная была обставлена просто, но по-деловому.","[Приемная, была, обставлена, просто, ,, но, по...","[приемная, быть, обставить, просто, ,, но, по-...","[NOUN, AUX, VERB, ADV, PUNCT, CCONJ, ADV, PUNCT]","[None, None, None, None, None, None, None, None]","[{'Animacy': 'Inan', 'Case': 'Nom', 'Gender': ...","[3, 3, 0, 3, 7, 7, 4, 3]","[nsubj:pass, aux:pass, root, advmod, punct, cc...","[None, None, None, {'SpaceAfter': 'No'}, None,..."


In [15]:
# Проверка на соответствие количества слов и количества меток к ним
for i in range(len(train_df)):
    row = train_df.iloc[i, :]
    if len(row['source_words']) != len(row['upos']):
        print(f'ne at i = {i}')

In [16]:
tokenizer = train_bpe_tokenizer([CORPUS_TEXTS_FILEPATH], VOCABULARY_SIZE, MIN_FRECQUENCY_PAIR, unk_token=UNK_TOKEN, pad_token=PAD_TOKEN)
print(len(tokenizer.get_vocab()))

6212


In [17]:
MAX_WORDS_COUNT = max(find_max_words_source_len(train_df), find_max_words_source_len(validation_df))
MAX_TOKENS_COUNT = max(find_max_tokens_source_len(train_df, tokenizer), find_max_tokens_source_len(validation_df, tokenizer))
if ADD_BOS_EOS_TOKENS:
    MAX_TOKENS_COUNT += 2
    MAX_WORDS_COUNT += 2

print(MAX_WORDS_COUNT)
print(MAX_TOKENS_COUNT)

target_names = ['upos']
source_name = 'source_text'

205
332


In [18]:
# text = 'Что же ты делаешь? Ты, огузок, отклеивай наклейки от бананов!'
# text = 'Что же ты делаешь ? Ты , огузок , отклеивай наклейки от бананов !'
# tokens = tokenizer.encode(text).tokens
# tokens

In [19]:
source_vocab = Vocabulary(bos_token=BOS_TOKEN, eos_token=EOS_TOKEN, pad_token=PAD_TOKEN, mask_token=MASK_TOKEN, unk_token=UNK_TOKEN, add_bos_eos_tokens=ADD_BOS_EOS_TOKENS)
target_vocabs = {target_name: Vocabulary(bos_token=BOS_TOKEN, eos_token=EOS_TOKEN, pad_token=PAD_TOKEN,\
                                         mask_token=MASK_TOKEN, unk_token=UNK_TOKEN, add_bos_eos_tokens=ADD_BOS_EOS_TOKENS) for target_name in target_names}

for df in [train_df, validation_df, test_df]:
    for i in range(len(df)):
        source_vocab.add_tokens(tokenizer.encode(df[source_name].iloc[i]).tokens)
        for target_name in target_names:
            target_vocabs[target_name].add_tokens(df[target_name].iloc[i])
        

pad_idx = source_vocab.pad_idx
source_vocab_len = len(source_vocab)
cls_names_params = {key:len(target_vocabs[key]) for key in target_names}

In [20]:
letters_vocab = Vocabulary(bos_token=BOS_TOKEN, eos_token=EOS_TOKEN, pad_token=PAD_TOKEN, mask_token=MASK_TOKEN, unk_token=UNK_TOKEN, add_bos_eos_tokens=False)


for df in [train_df, validation_df, test_df]:
    for i in range(len(df)):
        for word in df['source_words'].iloc[i]:
            for char in word:
                letters_vocab.add_token(char)

letters_vocab_len = len(letters_vocab)

In [21]:
print(f'Количество батчей = {len(train_df)//BATCH_SIZE}')

print(f'Длина словаря токенов = {len(source_vocab)}')
print(f'Длина словаря символов = {letters_vocab_len}')
for key in target_names:
    print(f'Длина словаря признака {key} = {len(target_vocabs[key])}')

Количество батчей = 543
Длина словаря токенов = 5268
Длина словаря символов = 167
Длина словаря признака upos = 21


In [22]:
target_weights = {key : {} for key in target_names}

dataframes = [train_df, test_df, validation_df]

empty_nodes_counter = 0

for target_name in target_names:
    target_weights[target_name]['grammem_weight'] = 1.0 # Вес, вложимый конкретной граммемой в суммарную ошибку
    total_labels_count = 0
    labels_count = {}
    for df_idx, dataframe in enumerate(dataframes):
        for i in range(len(dataframe)):
            row = dataframe[target_name].iloc[i]
            for label in row:
                if label == '_':
                    # print(f'dataframe {df_idx}\n index {i}')
                    empty_nodes_counter += 1
                total_labels_count += 1
                if label not in labels_count:
                    labels_count[label] = 1
                else:
                    labels_count[label] += 1
    
    target_classes_weights = [1]*len(target_vocabs[target_name])
    if USE_CLASSES_WEIGHTS:
        for key in labels_count.keys():
            labels_count[key] = labels_count[key] / total_labels_count
            target_classes_weights[target_vocabs[target_name].token_to_idx[key]] = CLASSES_WEIGHTS_SCALER / (math.exp(CLASSES_WEIGHTS_SCALER * labels_count[key]))
    target_weights[target_name]['classes_weights'] = torch.tensor(target_classes_weights, dtype=torch.float32)

    print(f'target_weights {target_weights[target_name]['classes_weights']}')
    print(f'classes_distribution {labels_count}')

    print(empty_nodes_counter)

target_weights tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1.])
classes_distribution {'NOUN': 362393, 'PUNCT': 280655, 'ADJ': 140593, 'PROPN': 50180, 'AUX': 13740, 'VERB': 172851, 'ADP': 140972, 'ADV': 76404, 'CCONJ': 55088, 'PART': 50341, 'PRON': 66174, 'DET': 54745, 'SCONJ': 28529, 'NUM': 17909, '_': 2203, 'INTJ': 231, 'X': 3495, 'SYM': 1280}
2203


In [23]:
if USE_PRETRAINED:
    with open("data/train_states.json", "r", encoding="utf-8") as file:
        train_states = json.load(file)
        training_epochs = int(train_states[-1]['training_epochs'])

    with open("data/validation_states.json", "r", encoding="utf-8") as file:
        validation_states = json.load(file)
    
    model = torch.load(MODEL_SAVE_FILEPATH, weights_only=False)
else:
    train_states = []
    validation_states = []
    training_epochs = 0
    model = MHAModel(MAX_WORDS_COUNT, MAX_TOKENS_COUNT, MAX_WORD_SUBTOKENS_COUNT, MAX_LETTERS_COUNT, letters_vocab_len, source_vocab_len, TOKENS_EMBEDDING_DIM, LETTERS_EMBEDDING_DIM,\
                     MAIN_ATTENTION_DIM, MAIN_NUM_HEADS, MAIN_NUM_ENCODER_LAYERS, CLASSIFIER_FC_HIDDEN_DIM, MAIN_ENCODER_FC_HIDDEN_DIM,\
                     cls_names_params, WORDS_POS_ENCODING, TOKENS_POS_ENCODING, WORD_SUBTOKENS_POS_ENCODING, LETTERS_POS_ENCODING,\
                     SUBTOKENS_AGGREGATION, AGGREGATION_MOMENT, LETTERS_IN_WORD_ATTENTION_DIM, DROPOUT, TEMPERATURE, BATCH_FIRST, WORD_REPRESENTATION, INIT_WEIGHTS, BIAS, pad_idx)

In [24]:
vectorizer = Vectorizer(tokenizer, source_vocab, target_vocabs, letters_vocab, pad_idx)
dataset = CustomDataset(vectorizer, train_df, target_names, MAX_TOKENS_COUNT, MAX_WORDS_COUNT, MAX_LETTERS_COUNT, add_bos_eos_tokens=ADD_BOS_EOS_TOKENS, valid_df=validation_df)

model = model.to(device=DEVICE)
optimizer = optim.AdamW(model.parameters(), LEARNING_RATE, weight_decay=WEIGHT_DECAY)

In [25]:
try:
    for epoch in range(EPOCHS):
        training_epochs += 1
        print('-'*40)
        print(f'Epoch {training_epochs}')
        dataset.set_dataframe_split('train')
        batch_generator = generate_batches(dataset, BATCH_SIZE, SHUFFLE, DROP_LAST, DEVICE)
        epoch_sum_train_loss = 0.0
        epoch_running_train_loss = 0.0
        train_running_acc = {key:0.0 for key in target_names}
        model.train()
        for batch_idx, batch_dict in enumerate(batch_generator):

            optimizer.zero_grad()
            
            predictions = model(batch_dict['source_x'], batch_dict['subtokens_cnt'], batch_dict['letters'])

            total_loss, train_losses = compute_loss(predictions, batch_dict, target_names, target_weights, pad_idx)

            total_loss.backward()

            optimizer.step()

            # Средние потери и точность
            epoch_running_train_loss += (total_loss.item() - epoch_running_train_loss) / (batch_idx + 1)
            epoch_sum_train_loss += total_loss.item()

            acc_t = compute_accuracy(predictions, batch_dict, target_names, pad_idx)
            for key in target_names:
                train_running_acc[key] += (acc_t[key] - train_running_acc[key]) / (batch_idx + 1)

        dataset.set_dataframe_split('validation')
        batch_generator = generate_batches(dataset, BATCH_SIZE, SHUFFLE, DROP_LAST, DEVICE)
        epoch_sum_valid_loss = 0.0
        epoch_running_valid_loss = 0.0
        valid_running_acc = {key:0.0 for key in target_names}
        model.eval()

        with torch.no_grad():
            for batch_idx, batch_dict in enumerate(batch_generator):
                
                predictions = model(batch_dict['source_x'], batch_dict['subtokens_cnt'], batch_dict['letters'])

                total_loss, valid_losses = compute_loss(predictions, batch_dict, target_names, target_weights, pad_idx)

                # Средние потери и точность
                epoch_running_valid_loss += (total_loss.item() - epoch_running_valid_loss) / (batch_idx + 1)
                epoch_sum_valid_loss += total_loss.item()

                acc_t = compute_accuracy(predictions, batch_dict, target_names, pad_idx)
                for key in target_names:
                    valid_running_acc[key] += (acc_t[key] - valid_running_acc[key]) / (batch_idx + 1)

        train_losses_dict = {}
        valid_losses_dict = {}
        for key in train_losses.keys():
            train_losses_dict[key] = train_losses[key].item()
            valid_losses_dict[key] = valid_losses[key].item()

        train_states.append({'training_epochs' : training_epochs, 'epoch_summed_loss' : epoch_sum_train_loss,\
                             'epoch_running_loss' : epoch_running_train_loss, 'losses' : train_losses_dict, 'accuracy' : train_running_acc})
        validation_states.append({'training_epochs' : training_epochs, 'epoch_summed_loss' : epoch_sum_valid_loss,\
                                  'epoch_running_loss' : epoch_running_valid_loss, 'losses' : valid_losses_dict, 'accuracy' : valid_running_acc})
        
        print(f'Train: Суммарная ошибка эпохи {epoch_sum_train_loss}')
        print(f'Train: Средняя ошибка эпохи {epoch_running_train_loss}')
        for key in target_names:
            print(f'Train: Ошибка на признаке {key}: {train_losses_dict[key]}')
            print(f'Train: Точность на признаке {key}: {train_running_acc[key]*100}')

        print('-'*10)
        print(f'Validation: Суммарная ошибка эпохи {epoch_sum_valid_loss}')
        print(f'Validation: Средняя ошибка эпохи {epoch_running_valid_loss}')
        for key in target_names:
            print(f'Validation: Ошибка на признаке {key}: {valid_losses_dict[key]}')
            print(f'Validation: Точность на признаке {key}: {valid_running_acc[key]*100}')

except KeyboardInterrupt:
    print('Принудительная остановка')

----------------------------------------
Epoch 31
Train: Суммарная ошибка эпохи 8.011382440570742
Train: Средняя ошибка эпохи 0.014753927146539116
Train: Ошибка на признаке upos: 0.01969347894191742
Train: Точность на признаке upos: 99.43470192527343
----------
Validation: Суммарная ошибка эпохи 10.187348313629627
Validation: Средняя ошибка эпохи 0.14764272918303806
Validation: Ошибка на признаке upos: 0.13972297310829163
Validation: Точность на признаке upos: 97.15965965967898
----------------------------------------
Epoch 32
Train: Суммарная ошибка эпохи 7.487947619985789
Train: Средняя ошибка эпохи 0.013789958784504217
Train: Ошибка на признаке upos: 0.013740692287683487
Train: Точность на признаке upos: 99.47583162521794
----------
Validation: Суммарная ошибка эпохи 10.124381937086582
Validation: Средняя ошибка эпохи 0.1467301730012548
Validation: Ошибка на признаке upos: 0.20882944762706757
Validation: Точность на признаке upos: 97.16912215038093
----------------------------------

In [26]:
dfdf

NameError: name 'dfdf' is not defined

In [27]:
save_results_to_file(model, MODEL_SAVE_FILEPATH, train_states, validation_states)

In [ ]:
dataframes = [train_df, test_df, validation_df]

empty_nodes_counter = 0

for target_name in target_names:
    total_labels_count = 0
    labels_count = {}
    for df_idx, dataframe in enumerate(dataframes):
        for i in range(len(dataframe)):
            row = dataframe[target_name].iloc[i]
            for label in row:
                # if label == '_':
                #     print(f'dataframe {df_idx}\n index {i}')
                #     empty_nodes_counter += 1
                total_labels_count += 1
                if label not in labels_count:
                    labels_count[label] = 1
                else:
                    labels_count[label] += 1
    
    for key in labels_count.keys():
        labels_count[key] = labels_count[key] / total_labels_count * 100

    print(f'classes_distribution {labels_count}')

    # print(empty_nodes_counter)

classes_distribution {'NOUN': 23.876469824737793, 'PUNCT': 18.491115001288065, 'ADJ': 9.263050119812911, 'PROPN': 3.3061379657039245, 'AUX': 0.905267749078755, 'VERB': 11.388386877439, 'ADP': 9.288020751319523, 'ADV': 5.03392118636195, 'CCONJ': 3.6295043494359867, 'PART': 3.3167455426763905, 'PRON': 4.359911792397201, 'DET': 3.6069055984946465, 'SCONJ': 1.8796494624066813, 'NUM': 1.1799446956514863, '_': 0.14514591348038552, 'INTJ': 0.015219566960494353, 'X': 0.2302700715451418, 'SYM': 0.08433353120966568}


In [ ]:
# Исследование ошибок модели
try:
    mistake_count = 0
    with torch.no_grad():
        for idx in range(len(validation_df)):
            row = validation_df.iloc[idx]
            vectorized_dict = vectorizer.vectorize(row, target_names, MAX_TOKENS_COUNT, MAX_WORDS_COUNT, MAX_LETTERS_COUNT, ADD_BOS_EOS_TOKENS)

            # Тензоры для входа и подсчёта субтокенов
            source_x = torch.tensor(vectorized_dict['src_vectorized']).unsqueeze(0).to(DEVICE)
            subtokens_cnt = torch.tensor(vectorized_dict['subtokens_cnt']).unsqueeze(0).to(DEVICE)
            letters = torch.tensor(vectorized_dict['letters']).unsqueeze(0).to(DEVICE)

            # Тензоры целевых меток
            trg_tensors = {key: torch.tensor(vectorized_dict['trg_vectorized'][key]).unsqueeze(0).to(DEVICE)
                        for key in target_names}

            predictions = model(source_x, subtokens_cnt, letters)

            # Индексы предсказанных меток
            pred_indices = {}
            for key in target_names:
                _, idx_max = predictions[key].max(dim=-1)
                pred_indices[key] = idx_max.squeeze(0)

            # Ошибки
            mistakes = reveal_mistakes(predictions, trg_tensors, target_names, pad_idx)

            for key in target_names:
                mistake_tensor = mistakes[key].squeeze(0).float()
                if mistake_tensor.sum() > 0:
                    mistake_count += 1
                    print('-'*30)
                    print(f'{mistake_count}. Исходное предложение: {row["source_text"]}. Индекс в датафрейме = {idx}. Количество ошибок: {mistake_tensor.sum()}')
                    for i in range(mistake_tensor.size(0)):
                        if mistake_tensor[i] > 0:
                            word = row['source_words'][i]
                            windowed_word = ' '.join(row['source_words'][i-1:i+2])
                            predicted_label = target_vocabs[key].idx_to_token[pred_indices[key][i].item()]
                            print(f'Ошибка в " {windowed_word} ".  Слово " {word} ", граммема {key}')
                            print(f'Токенизированное слово: {tokenizer.encode(word).tokens}')
                            print(f'Предсказанная метка: {predicted_label}, реальная метка: {row[key][i]}')
                            print(f'Пропорция предсказанной метки {labels_count[predicted_label]}, Пропорция реальной метки {labels_count[row[key][i]]}')
except KeyboardInterrupt:
    print('Принудительная остановка')

------------------------------
1. Исходное предложение: Алгоритм, от имени учёного аль-Хорезми, - точный набор инструкций, описывающих порядок действий исполнителя для достижения результата решения задачи за конечное время.. Индекс в датафрейме = 0. Количество ошибок: 2.0
Ошибка в " имени учёного аль ".  Слово " учёного ", граммема upos
Токенизированное слово: ['уч', '##ё', '##ного']
Предсказанная метка: ADJ, реальная метка: NOUN
Пропорция предсказанной метки 9.263050119812911, Пропорция реальной метки 23.876469824737793
Ошибка в " учёного аль - ".  Слово " аль ", граммема upos
Токенизированное слово: ['аль']
Предсказанная метка: ADV, реальная метка: PART
Пропорция предсказанной метки 5.03392118636195, Пропорция реальной метки 3.3167455426763905
------------------------------
2. Исходное предложение: Это связано с тем, что работа каких-то инструкций алгоритма может быть зависима от других инструкций или результатов их работы.. Индекс в датафрейме = 2. Количество ошибок: 1.0
Ошибка в " 